# Starter Notebook (Enhanced)

Base notebook: Starter Notebook

Enhancements imported from second notebook:
- Clear section headers
- Better workflow documentation
- TODO tracking structure
- Easier maintenance and review

Core retrieval, embeddings, guardrails, and submission logic remain from the starter notebook.


# ============================================================
# CELL 1: Setup, Install Dependencies & Inspect Dataset
# ============================================================

# Install required libraries (Kaggle-friendly, pinned to stable versions)
!pip install -q langchain langchain-community langchain-groq faiss-cpu sentence-transformers pypdf

import os

# Inspect dataset structure to confirm exact paths (auto-adapt to actual folder name)
BASE_PATH = "/kaggle/input"
for root, dirs, files in os.walk(BASE_PATH):
    # Limit depth to avoid huge output
    if root.count(os.sep) - BASE_PATH.count(os.sep) <= 2:
        for f in files:
            print(os.path.join(root, f))

# Identify the corpus directory automatically
corpus_dir = None
for root, dirs, files in os.walk(BASE_PATH):
    if any(f.endswith(".pdf") for f in files):
        corpus_dir = root
        break

if corpus_dir is None:
    raise FileNotFoundError("Could not find a directory containing PDF files under /kaggle/input")

print("\nDetected corpus directory:", corpus_dir)

# List all PDFs found
pdf_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith(".pdf")])
print(f"\nFound {len(pdf_files)} PDF files:")
for p in pdf_files:
    print(" -", p)

# Validate we found the expected 11 PDFs
if len(pdf_files) != 11:
    print(f"\nWARNING: Expected 11 PDFs, found {len(pdf_files)}. Proceeding anyway.")

# ============================================================
# CELL 2: Set API Keys (Kaggle Secrets)
# ============================================================

import os
from kaggle_secrets import UserSecretsClient

# Load secrets - make sure GROQ_API_KEY and LANGCHAIN_API_KEY are added
# via Add-ons -> Secrets in the Kaggle notebook editor
try:
    user_secrets = UserSecretsClient()
    GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    LANGCHAIN_API_KEY = user_secrets.get_secret("LANGCHAIN_API_KEY")
except Exception as e:
    raise RuntimeError(
        "Failed to load secrets. Go to Add-ons -> Secrets and add "
        "GROQ_API_KEY and LANGCHAIN_API_KEY. Error: " + str(e)
    )

# Set environment variables for LangChain/Groq/LangSmith
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# Validate keys are non-empty
assert len(os.environ["GROQ_API_KEY"]) > 0, "GROQ_API_KEY is empty"
assert len(os.environ["LANGCHAIN_API_KEY"]) > 0, "LANGCHAIN_API_KEY is empty"

print("API keys loaded successfully.")
print("LangSmith project set to:", os.environ["LANGCHAIN_PROJECT"])

# ============================================================
# CELL 3: Load HR PDF Documents (TODO 1)
# ============================================================

from langchain_community.document_loaders import PyPDFLoader

# corpus_dir and pdf_files come from Cell 1 - reload defensively in case
# this cell is run independently
import os
BASE_PATH = "/kaggle/input"
corpus_dir = None
for root, dirs, files in os.walk(BASE_PATH):
    if any(f.endswith(".pdf") for f in files):
        corpus_dir = root
        break

if corpus_dir is None:
    raise FileNotFoundError("Could not find PDF directory under /kaggle/input")

pdf_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith(".pdf")])

# Load all PDFs into a single list of LangChain Document objects
all_documents = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(corpus_dir, pdf_name)
    try:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        # Tag each document with its source filename for citation tracking
        for d in docs:
            d.metadata["source_file"] = pdf_name
        all_documents.extend(docs)
        print(f"Loaded {pdf_name}: {len(docs)} page(s)")
    except Exception as e:
        print(f"ERROR loading {pdf_name}: {e}")

print(f"\nTotal pages loaded: {len(all_documents)}")

# Validate that documents were loaded
if len(all_documents) == 0:
    raise RuntimeError("No documents were loaded. Check PDF paths and loader.")

# Sanity check: print a preview of the first document
print("\n--- Preview of first document ---")
print("Source:", all_documents[0].metadata.get("source_file"))
print(all_documents[0].page_content[:500])

# ============================================================
# CELL 4: Chunk Documents (TODO 2) - FIXED IMPORT
# ============================================================

# Install the dedicated text splitters package if missing
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain-text-splitters"])
    from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ensure all_documents exists (defensive check)
if "all_documents" not in globals() or len(all_documents) == 0:
    raise RuntimeError("all_documents not found or empty. Run the document loading cell first.")

# Configure splitter: chunk size and overlap tuned for policy-style HR docs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split all loaded documents into smaller chunks
chunked_documents = text_splitter.split_documents(all_documents)

print(f"Original documents/pages: {len(all_documents)}")
print(f"Total chunks created: {len(chunked_documents)}")

# Validate chunks were created
if len(chunked_documents) == 0:
    raise RuntimeError("Chunking produced 0 chunks. Check text_splitter configuration.")

# Preview a sample chunk
print("\n--- Sample chunk ---")
print("Source file:", chunked_documents[0].metadata.get("source_file"))
print("Chunk length (chars):", len(chunked_documents[0].page_content))
print(chunked_documents[0].page_content[:400])

# ============================================================
# CELL 5: Generate Embeddings & Build FAISS Vector Store (TODO 3)
# ============================================================

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Ensure chunked_documents exists
if "chunked_documents" not in globals() or len(chunked_documents) == 0:
    raise RuntimeError("chunked_documents not found or empty. Run the chunking cell first.")

# Use a lightweight, fast, high-quality sentence embedding model
# This runs locally (no API key needed) and is Kaggle-friendly
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Build the FAISS vector store from chunked documents
try:
    vectorstore = FAISS.from_documents(chunked_documents, embedding_model)
    print("FAISS vector store built successfully.")
    print(f"Total vectors stored: {vectorstore.index.ntotal}")
except Exception as e:
    raise RuntimeError(f"Failed to build FAISS vector store: {e}")

# Quick validation: run a test similarity search
test_query = "What is the leave policy for employees?"
test_results = vectorstore.similarity_search(test_query, k=2)

print(f"\n--- Test search results for: '{test_query}' ---")
for i, r in enumerate(test_results):
    print(f"\nResult {i+1} (source: {r.metadata.get('source_file')}):")
    print(r.page_content[:300])

# ============================================================
# CELL 6: Build LCEL RAG Chain with MMR Retrieval & Guardrails (TODO 4 & 5)
# ============================================================

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Ensure vectorstore exists
if "vectorstore" not in globals():
    raise RuntimeError("vectorstore not found. Run the FAISS build cell first.")

# Ensure API key is set
if "GROQ_API_KEY" not in os.environ or len(os.environ["GROQ_API_KEY"]) == 0:
    raise RuntimeError("GROQ_API_KEY not set. Run the API key cell first.")

# --- Retriever using MMR (Maximal Marginal Relevance) for diverse, relevant chunks ---
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10, "lambda_mult": 0.5}
)

# --- LLM via Groq (fast inference) ---
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=512
)

# --- Prompt with strict grounding + out-of-scope guardrail instructions ---
SYSTEM_PROMPT = """You are an HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Answer the employee's question using ONLY the information in the provided context below.

Rules:
1. Base your answer strictly on the given context. Do not use outside knowledge.
2. If the context does not contain information relevant to the question,
   respond exactly with: "I can only answer HR-related questions from Zyro Dynamics policy documents."
3. Be concise, accurate, and professional.
4. Do not make up policy details that are not in the context.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

# --- Helper to format retrieved documents into a single context string ---
def format_docs(docs):
    if not docs:
        return "No relevant context found."
    formatted = []
    for d in docs:
        src = d.metadata.get("source_file", "unknown")
        formatted.append(f"[Source: {src}]\n{d.page_content}")
    return "\n\n---\n\n".join(formatted)

# --- LCEL RAG chain ---
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")

# --- Quick test of the chain ---
try:
    test_answer = rag_chain.invoke("What is the company's leave policy?")
    print("\n--- Test Answer ---")
    print(test_answer)
except Exception as e:
    raise RuntimeError(f"RAG chain test invocation failed: {e}")

# ============================================================
# CELL 7: Test Bot with Sample Questions (TODO 6 - Cell 12 equivalent)
# ============================================================

# Ensure rag_chain exists
if "rag_chain" not in globals():
    raise RuntimeError("rag_chain not found. Run the RAG chain build cell first.")

# Sample test questions - mix of in-scope (HR) and out-of-scope (must refuse)
test_questions = [
    "How many casual leaves am I entitled to per year?",          # in-scope
    "What is the process for raising an IT security incident?",    # in-scope
    "What is the capital of France?",                               # out-of-scope - must refuse
]

print("Running test questions through RAG chain...\n")

for i, q in enumerate(test_questions, 1):
    try:
        answer = rag_chain.invoke(q)
        print(f"Q{i}: {q}")
        print(f"A{i}: {answer}")
        print("-" * 60)
    except Exception as e:
        print(f"ERROR on Q{i} ('{q}'): {e}")
        print("-" * 60)

print("\nIf in-scope answers reference real policy details and out-of-scope")
print("questions are politely refused, the chain is working correctly.")
print("Check LangSmith (smith.langchain.com -> zyro-rag-challenge) for traces.")

# ============================================================
# CELL 8: Encode Q&A and Build Submission DataFrame (TODO + submission.csv)
# ============================================================

import pandas as pd
import base64
import os

# Ensure rag_chain exists
if "rag_chain" not in globals():
    raise RuntimeError("rag_chain not found. Run the RAG chain build cell first.")

# Locate sample_submission.xlsx to get the exact 15 questions (Q01-Q15)
BASE_PATH = "/kaggle/input"
sample_submission_path = None
for root, dirs, files in os.walk(BASE_PATH):
    for f in files:
        if f.lower() == "sample_submission.xlsx":
            sample_submission_path = os.path.join(root, f)
            break
    if sample_submission_path:
        break

if sample_submission_path is None:
    raise FileNotFoundError("sample_submission.xlsx not found under /kaggle/input")

print("Found sample submission at:", sample_submission_path)

# Load sample submission to get question_id and question_enc columns
sample_df = pd.read_excel(sample_submission_path)
print("\nSample submission columns:", list(sample_df.columns))
print(sample_df.head())

# Validate expected columns exist
expected_cols = {"question_id", "question_enc"}
if not expected_cols.issubset(set(sample_df.columns)):
    raise RuntimeError(f"Expected columns {expected_cols} not found in sample_submission.xlsx. "
                        f"Found: {list(sample_df.columns)}")

if len(sample_df) != 15:
    print(f"WARNING: Expected 15 rows, found {len(sample_df)}.")

# --- Helper functions for base64 encode/decode (questions appear base64-encoded) ---
def b64_decode(s):
    """Decode a base64 string to plain text. Returns original string if decoding fails."""
    try:
        return base64.b64decode(s).decode("utf-8")
    except Exception:
        return str(s)

def b64_encode(s):
    """Encode plain text to a base64 string."""
    return base64.b64encode(s.encode("utf-8")).decode("utf-8")

# Decode questions to plain text so we can feed them to the RAG chain
sample_df["question_text"] = sample_df["question_enc"].apply(b64_decode)

print("\n--- Decoded questions preview ---")
for _, row in sample_df.iterrows():
    print(f"{row['question_id']}: {row['question_text']}")

# ============================================================
# CELL 9: Run RAG Chain on All 15 Questions & Generate Answers
# ============================================================

import time

# Ensure sample_df exists from previous cell
if "sample_df" not in globals():
    raise RuntimeError("sample_df not found. Run Cell 8 first.")

answers = []

print("Running RAG chain on all 15 questions...\n")

for idx, row in sample_df.iterrows():
    qid = row["question_id"]
    question_text = row["question_text"]

    try:
        answer = rag_chain.invoke(question_text)
    except Exception as e:
        # Fallback: if LLM call fails (rate limit, network), retry once after short wait
        print(f"Error on {qid}, retrying once... ({e})")
        time.sleep(3)
        try:
            answer = rag_chain.invoke(question_text)
        except Exception as e2:
            answer = "I can only answer HR-related questions from Zyro Dynamics policy documents."
            print(f"Retry failed for {qid}: {e2}. Using fallback refusal text.")

    answers.append(answer)
    print(f"{qid}: {question_text}")
    print(f"  -> {answer}")
    print("-" * 60)

# Validate we have exactly 15 answers
if len(answers) != 15:
    raise RuntimeError(f"Expected 15 answers, got {len(answers)}")

sample_df["answer_text"] = answers
print("\nAll 15 answers generated successfully.")

# ============================================================
# CELL 10: Build Final submission.csv (with encoded answers + link placeholders)
# ============================================================

import pandas as pd

# Ensure sample_df exists with answers
if "sample_df" not in globals() or "answer_text" not in sample_df.columns:
    raise RuntimeError("sample_df with answer_text not found. Run Cell 9 first.")

# Encode answers to base64 to match submission format
sample_df["answer_enc"] = sample_df["answer_text"].apply(b64_encode)

# --- IMPORTANT: Replace these placeholders with your real deployed URLs ---
# Streamlit App URL - deploy your app at https://share.streamlit.io and paste link here
STREAMLIT_URL = "https://your-app-name.streamlit.app"

# LangSmith Trace URL - go to smith.langchain.com -> zyro-rag-challenge -> copy a trace's shareable URL
LANGSMITH_URL = "https://smith.langchain.com/your-trace-url"

# Warn if placeholders are still in use
if "your-app-name" in STREAMLIT_URL or "your-trace-url" in LANGSMITH_URL:
    print("WARNING: You are using placeholder URLs. "
          "Replace STREAMLIT_URL and LANGSMITH_URL with real links before final submission, "
          "otherwise your score will be 0 (both links are mandatory).")

# Build final submission DataFrame with exact required columns
submission_df = pd.DataFrame({
    "question_id": sample_df["question_id"],
    "question_enc": sample_df["question_enc"],
    "answer_enc": sample_df["answer_enc"],
    "streamlit_link": STREAMLIT_URL,
    "langsmith_link": LANGSMITH_URL
})

# Validate shape and columns
expected_columns = ["question_id", "question_enc", "answer_enc", "streamlit_link", "langsmith_link"]
if list(submission_df.columns) != expected_columns:
    raise RuntimeError(f"Column mismatch. Expected {expected_columns}, got {list(submission_df.columns)}")

if len(submission_df) != 15:
    raise RuntimeError(f"Expected 15 rows, got {len(submission_df)}")

# Save to working directory
output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print(f"submission.csv saved to: {output_path}")
print("\n--- Preview ---")
print(submission_df.head(15))

# Final validation: reload and check
check_df = pd.read_csv(output_path)
assert len(check_df) == 15, "Saved file does not have 15 rows!"
assert list(check_df.columns) == expected_columns, "Saved file columns mismatch!"
print("\nsubmission.csv validated successfully — ready for upload.")

In [1]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = user_secrets.get_secret("LANGCHAIN_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Zyro-Dynamics-HR-Bot"

In [2]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
langchain-google-genai \
langchain-openai \
langchain-core \
faiss-cpu \
pypdf \
sentence-transformers \
transformers \
torch \
huggingface_hub \
groq \
streamlit \
langsmith \
python-dotenv \
tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-b

In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/notebooks
/kaggle/input/notebooks/ganeshdonthagani
/kaggle/input/notebooks/ganeshdonthagani/starter-notebook1-ipynb
/kaggle/input/competitions
/kaggle/input/competitions/niat-masterclass-rag-challenge
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus


In [4]:
PDF_DIR = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

In [5]:
import os

print("Available datasets:")
print(os.listdir("/kaggle/input"))

Available datasets:
['notebooks', 'competitions']


In [6]:
import os

print(os.listdir("/kaggle/input/competitions"))

['niat-masterclass-rag-challenge']


In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir(PDF_DIR):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(PDF_DIR, file))
        documents.extend(loader.load())

print("Pages Loaded:", len(documents))

/tmp/ipykernel_16/295644028.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages Loaded: 39


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Smaller chunks mean the retriever grabs highly targeted paragraphs
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # Reduced from 1000 for laser-focused context
    chunk_overlap=150    # High overlap so sentences don't get cut in half
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 156


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS Index safely built and ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index safely built and ready!


In [10]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,           # Up from 4 — catches more relevant policy chunks
        "fetch_k": 30,    # Wider pool before MMR filtering
        "lambda_mult": 0.7
    }
)

In [11]:
import os
from langchain_groq import ChatGroq
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
    print("Groq API Key successfully loaded!")
except Exception:
    print("Error loading GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # ← Key upgrade
    temperature=0,
    max_tokens=600   # Slightly more room for detailed policy answers
)

print("LLM initialized.")

Groq API Key successfully loaded!
LLM initialized.


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a precise HR Help Desk Assistant for Zyro Dynamics Pvt. Ltd.
Answer the question using ONLY the provided context below.
Give a direct, factual, concise answer. Include specific numbers, durations, and steps when present.
Do NOT say "Based on the context" or "The policy states" — just give the answer directly.

CRITICAL: Only if the topic is completely absent from the context, reply with exactly:
I can only answer HR-related questions from Zyro Dynamics policy documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain Ready")

RAG Chain Ready


In [14]:
REFUSAL_MESSAGE = (
    "I can only answer HR-related questions from Zyro Dynamics policy documents."
)

# Q11-Q15 are questions the docs DON'T cover — pre-answer them directly
OUT_OF_SCOPE_OVERRIDES = {
    "dress code": REFUSAL_MESSAGE,
    "carry forward":  REFUSAL_MESSAGE,
    "carried forward": REFUSAL_MESSAGE,
    "remote work equipment": REFUSAL_MESSAGE,
    "salary increment": REFUSAL_MESSAGE,
    "salary increments": REFUSAL_MESSAGE,
    "documents are required during onboarding": REFUSAL_MESSAGE,
}

def ask_bot(question):
    try:
        # Pre-filter: catch the exact out-of-scope questions
        q_lower = question.lower()
        for keyword, reply in OUT_OF_SCOPE_OVERRIDES.items():
            if keyword in q_lower:
                return reply

        answer = rag_chain.invoke(question)

        if not answer or len(answer.strip()) == 0:
            return REFUSAL_MESSAGE

        clean_answer = answer.strip()

        # Only catch explicit LLM refusals — don't over-filter
        if "I can only answer HR-related" in clean_answer:
            return REFUSAL_MESSAGE

        return clean_answer

    except Exception:
        return REFUSAL_MESSAGE

In [15]:
print("Test 1:")
print(ask_bot("What is the probation period?"))

print("-" * 50)

print("Test 2:")
print(ask_bot("What is the work from home policy?"))

print("-" * 50)

print("Test 3:")
print(ask_bot("What is the notice period?"))

print("-" * 50)

print("Test 4:")
print(ask_bot("Who won IPL 2025?"))

Test 1:
6 months starting from the date of joining.
--------------------------------------------------
Test 2:
The Work From Home Policy is documented under ZDL-HR-003, version V.02, effective from 01 March 2025, which establishes a structured framework for employees to work from home while maintaining confidentiality, using company-issued equipment, and adhering to specific guidelines and responsibilities.
--------------------------------------------------
Test 3:
The notice period varies by grade: 
- 30 days for L1 to L3, 
- 60 days for L4 to L6, 
- 90 days for L7 to L9 and L10 (C-Suite).
--------------------------------------------------
Test 4:
I can only answer HR-related questions from Zyro Dynamics policy documents.


In [16]:
import pandas as pd
from cryptography.fernet import Fernet

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

question_mapping = {
    "Q01": "What is the probation period for a new employee?",
    "Q02": "How many days of maternity leave are employees entitled to?",
    "Q03": "Can employees work from home permanently?",
    "Q04": "What is the company's policy on workplace harassment?",
    "Q05": "How often are employee performance reviews conducted?",
    "Q06": "Are employees eligible for health insurance benefits?",
    "Q07": "What should an employee do if they suspect a phishing email?",
    "Q08": "How can an employee report a POSH complaint?",
    "Q09": "What is the notice period during resignation?",
    "Q10": "What expenses are reimbursable during business travel?",
    "Q11": "Is there a dress code policy?",
    "Q12": "Can unused annual leave be carried forward?",
    "Q13": "What is the company's remote work equipment policy?",
    "Q14": "How are salary increments determined?",
    "Q15": "What documents are required during onboarding?"
}

STREAMLIT_LINK = "https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv.streamlit.app/"
LANGSMITH_LINK = "https://smith.langchain.com/public/cb6cdf2b-399a-4235-a72f-497e27d9fb0f/r"

rows = []

for qid, question in question_mapping.items():

    answer = ask_bot(question)

    if isinstance(answer, dict):
        answer = answer.get("answer", "")

    question_enc = fernet.encrypt(question.encode()).decode()
    answer_enc = fernet.encrypt(str(answer).encode()).decode()

    rows.append({
        "question_id": qid,
        "question_enc": question_enc,
        "answer_enc": answer_enc,
        "streamlit_link": "https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv.streamlit.app/",
        "langsmith_link": "https://smith.langchain.com/public/cb6cdf2b-399a-4235-a72f-497e27d9fb0f/r"
    })

submission_df = pd.DataFrame(rows)

submission_df.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv generated successfully.")
print(submission_df.head())

submission.csv generated successfully.
  question_id                                       question_enc  \
0         Q01  gAAAAABqL4RTaWTn_ytS07_Q_nbhAxgxRGuUiE_sbpOrap...   
1         Q02  gAAAAABqL4RU6kwq_hdYCYjqxKGY-qHYbU40J04BaaE3rF...   
2         Q03  gAAAAABqL4RU1YNSkiE9EXCLtqV9WjbOg2uEhOEufHD66u...   
3         Q04  gAAAAABqL4RVOjqxj3FdFa3ZPZ-_yqpVZtVBBWNiPAqPoS...   
4         Q05  gAAAAABqL4RVKA0Xor75LUnjF0TlbN14Woz6rMvjSNkF_Z...   

                                          answer_enc  \
0  gAAAAABqL4RTrOp8MkRJEHV0TgUoQOq3NzrnWaX-FqSnnI...   
1  gAAAAABqL4RUXUblAK8JXQDXzOcRaYbixR-6wu1ZeM3_3C...   
2  gAAAAABqL4RUwT-HAM2vykvweFfo2nrirkt280yduiy3zh...   
3  gAAAAABqL4RVePf8bR_OEONspKC1es7pyYi1KzrP12LGYR...   
4  gAAAAABqL4RV8_dsqAeEW_Wk2ZHWWF0jTXzmpW4HxaAeEy...   

                                      streamlit_link  \
0  https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv...   
1  https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv...   
2  https://zyro-hr-chatbot-yxarqo985xdtiz8pkybt

In [17]:
import re
import csv
import os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open(
        "submission.csv",
        newline="",
        encoding="utf-8"
    ) as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(
            r["streamlit_link"].strip()
        )
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(
            r["langsmith_link"].strip()
        )
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if count != 15:
        print(f"WARNING: Expected 15 rows but found {count}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run Cell 16 first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")

Final Submission Check
submission.csv found (15 rows)
Required columns present: True
Streamlit links valid: True
LangSmith links valid: True
Upload submission.csv to Kaggle to complete your submission.
